# 🏆 Fine-Tuning Granite-3.1-3B for Competitive Programming (Zip Upload Version)
This notebook fine-tunes the **IBM Granite 3.1 3B A800M Instruct** model using a dataset uploaded to Google Drive.

**Prerequisites:**
1. Download the dataset as a ZIP file from Kaggle.
2. Rename it to `codeforces_dataset.zip`.
3. Upload it to your Google Drive folder: `PredicAI_Training`.

In [ ]:
# @title 1: Install Dependencies & Setup Environment
import os
# CRITICAL: Helps debug errors if they happen again
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/PredicAI_Training"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints_codeforces_strict")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets tqdm

Mounted at /content/drive
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jxho11yo/unsloth_ca3af5353a764e3da5fc66bc1cf8e52a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jxho11yo/unsloth_ca3af5353a764e3da5fc66bc1cf8e52a
  Resolved https://github.com/unslothai/unsloth.git to commit 46f9be3dd1673ce38d662c25de005e233056c21b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.

In [ ]:
# @title 2: Load "SolvPredic" Dataset from Drive
import os
from datasets import load_dataset

# --- CONFIGURATION ---
DRIVE_ROOT = "/content/drive/MyDrive/PredicAI_Training"
FILE_NAME = "SolvPredic_Dataset.jsonl"
DATASET_PATH = os.path.join(DRIVE_ROOT, FILE_NAME)

# 1. Verify File Exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"❌ Dataset not found at: {DATASET_PATH}\n"
        f"Please check if you renamed the file correctly in Google Drive."
    )

print(f"📂 Found dataset: {DATASET_PATH}")
print("⏳ Loading data into memory...")

# 2. Load Dataset
# We use the 'json' loader which is optimized for .jsonl files
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"✅ Success! Loaded {len(dataset)} training samples.")

# 3. Preprocessing Check (Crucial step!)
# We check if the data has the 'messages' list required for ChatML
print("\n🔍 Inspecting data format...")
try:
    sample = dataset[0]
    if "messages" in sample and isinstance(sample["messages"], list):
        print("✅ Data Format is Correct: Found 'messages' column.")
        print("   Sample Role 1:", sample["messages"][0]['role']) # Should be 'system'
        print("   Sample Role 2:", sample["messages"][1]['role']) # Should be 'user'
    else:
        print("⚠️ WARNING: Dataset loaded, but 'messages' format seems wrong.")
        print(f"   Found columns: {dataset.column_names}")
except Exception as e:
    print(f"⚠️ Error inspecting data: {e}")

# Save this variable for the Trainer
train_dataset = dataset

In [ ]:
# @title 3: Load Qwen 2.5 Coder
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
import os

# Qwen 2.5 Coder is SOTA for 3B models
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
max_seq_length = 2048

print(f"⏳ Loading {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Setup LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Apply Chat Template (Standard ChatML)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

print("✅ Qwen 2.5 Coder Loaded Successfully.")

In [ ]:
# @title 4: Format Dataset
def format_chat_template(row):
    text = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

print("⏳ Formatting dataset...")
# Ensure train_dataset is loaded from Cell 2
train_dataset = train_dataset.map(format_chat_template)
print("✅ Done.")

In [ ]:
# @title 5: Start Training
import psutil
import builtins
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# Fix psutil error
builtins.psutil = psutil

# New Checkpoint Directory
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints_solvpredic_qwen")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 200,

        # 3 Epochs for 18k samples
        max_steps = 6900,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        logging_steps = 100,
        save_steps = 250,
        output_dir = CHECKPOINT_DIR,
        save_total_limit = 2,
        optim = "paged_adamw_32bit",
        gradient_checkpointing = True,
        weight_decay = 0.01,
        report_to="wandb",
        ignore_data_skip=True,
    ),
)

print(f"🚀 Starting Training on {len(train_dataset)} samples!")

if get_last_checkpoint(CHECKPOINT_DIR):
    print("🔄 Resuming from checkpoint...")
    trainer.train(resume_from_checkpoint=True)
else:
    print("✨ Starting FRESH training...")
    trainer.train()

In [ ]:
# @title 7: Export to GGUF
HF_TOKEN = "" # Your Token
REPO_ID = "duttaturja/SolvPredic-3B"  # Your desired repo name

print(f"💾 Pushing to Hub: {REPO_ID}...")
model.push_to_hub_gguf(
    REPO_ID,
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN
)
print("✅ DONE! Model is live on Hugging Face.")